# Star Cluster Injection Pipeline - RSP

This notebook provides a complete interface for:
1. **Configuring** injection parameters (smooth vs discrete stars)
2. **Injecting** synthetic star clusters into Rubin coadds
3. **Generating** postage stamps for each cluster
4. **Running** source detection and completeness analysis

## Setup

Clone the pipeline to RSP:
```bash
cd ~
git clone https://github.com/whosneha/INJECT.git
```

---
## 1. Setup and Imports

In [ ]:
# Setup pipeline path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Get RSP username and set pipeline path
USERNAME = os.environ.get('USER', 'your_username')
PIPELINE_PATH = f'/home/{USERNAME}/INJECT/star-cluster-injection-pipeline'

# Check if pipeline exists
if os.path.exists(PIPELINE_PATH):
    print(f"✓ Pipeline found at: {PIPELINE_PATH}")
    sys.path.insert(0, PIPELINE_PATH)
else:
    print(f"✗ Pipeline NOT found at: {PIPELINE_PATH}")
    print("\nRun in terminal:")
    print("  cd ~")
    print("  git clone https://github.com/whosneha/INJECT.git")

In [ ]:
# LSST imports
from lsst.daf.butler import Butler
from lsst.geom import Point2D

# Pipeline imports
from src.inject import inject_cluster, create_injection_catalog, inject_from_catalog
from src.light_profiles import PlummerProfile, KingProfile, mag_to_flux
from src.cluster_models import create_cluster, DiscreteStarCluster
from src.psf_convolution import get_psf_from_coadd, get_psf_fwhm_from_coadd
from src.visualization import plot_postage_stamps, plot_stamp_grid, plot_injection_summary
from src.retrieval import ClusterRetrieval, run_source_detection

print("✓ All imports successful!")

---
## 2. Load Coadd Image

Update the Butler configuration for your data.

In [ ]:
# ============ CONFIGURE DATA ACCESS ============
# UPDATE THESE FOR YOUR DATA

REPO = 'dp02'
COLLECTION = '2.2i/runs/DP0.2'

DATA_ID = {
    'tract': 4431,
    'patch': 17,
    'band': 'i'
}

print(f"Loading data from: {REPO}")
print(f"Collection: {COLLECTION}")
print(f"Data ID: {DATA_ID}")

In [ ]:
# Load Butler and coadd
print("\nInitializing Butler...")
butler = Butler(REPO, collections=COLLECTION)

print(f"Loading coadd...")
exposure = butler.get('deepCoadd', dataId=DATA_ID)

# Extract image array
image = exposure.image.array.copy()

print(f"\n✓ Coadd loaded successfully!")
print(f"  Shape: {image.shape}")
print(f"  Dtype: {image.dtype}")
print(f"  Range: [{image.min():.1f}, {image.max():.1f}]")

In [ ]:
# Display the coadd
fig, ax = plt.subplots(figsize=(10, 10))
vmin, vmax = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title(f"Coadd: tract={DATA_ID['tract']}, patch={DATA_ID['patch']}, band={DATA_ID['band']}")
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
plt.tight_layout()
plt.show()

---
## 3. Configuration

### Choose your injection method:
- **`smooth`**: Extended source with analytical profile (fast)
- **`discrete`**: Individual stars with IMF (realistic)

In [ ]:
# ============================================================
# INJECTION CONFIGURATION - MODIFY THIS CELL
# ============================================================

# Choose method: 'smooth' or 'discrete'
METHOD = 'smooth'  # <-- CHANGE THIS

# Number of clusters to inject
N_CLUSTERS = 20

# Magnitude range (total integrated magnitude)
MAG_MIN = 19.0
MAG_MAX = 25.0

# Half-light radius range (in pixels, 1 px = 0.2 arcsec)
R_HALF_MIN = 3.0   # 0.6 arcsec
R_HALF_MAX = 25.0  # 5.0 arcsec

# Profile type: 'plummer', 'king', 'eff', 'sersic'
PROFILE_TYPE = 'plummer'

# ---- Discrete star parameters (only used if METHOD='discrete') ----
N_STARS_MIN = 100
N_STARS_MAX = 500
IMF = 'kroupa'  # 'kroupa', 'chabrier', 'salpeter'
AGE_GYR_MIN = 0.5
AGE_GYR_MAX = 5.0

# Other settings
ADD_NOISE = True
EDGE_BUFFER = 150  # Stay away from edges (PSF issues)
SEED = 42

print(f"Configuration:")
print(f"  Method: {METHOD}")
print(f"  N clusters: {N_CLUSTERS}")
print(f"  Magnitude: [{MAG_MIN}, {MAG_MAX}]")
print(f"  r_half: [{R_HALF_MIN}, {R_HALF_MAX}] pixels")
print(f"  Profile: {PROFILE_TYPE}")
if METHOD == 'discrete':
    print(f"  N stars: [{N_STARS_MIN}, {N_STARS_MAX}]")
    print(f"  IMF: {IMF}")

### Option B: Create a custom configuration

In [ ]:
# Generate injection catalog
catalog = pipeline.generate_catalog()

# Display catalog summary
print(f"\nCatalog Summary:")
print(f"  N clusters: {len(catalog)}")
print(f"  Mag range: [{min(c['magnitude'] for c in catalog):.1f}, {max(c['magnitude'] for c in catalog):.1f}]")
print(f"  r_half range: [{min(c['r_half'] for c in catalog):.1f}, {max(c['r_half'] for c in catalog):.1f}] px")

In [ ]:
# Run injection
injected_image, injection_info = pipeline.run_injection()

In [ ]:
# Visualize injection
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(pipeline.image, [1, 99])

axes[0].imshow(pipeline.image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
axes[0].set_title('Original')

axes[1].imshow(injected_image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
for c in catalog:
    axes[1].scatter(c['x'], c['y'], s=50, facecolors='none', edgecolors='red', lw=1)
axes[1].set_title(f'With {len(catalog)} Injected Clusters')

diff = injected_image - pipeline.image
im = axes[2].imshow(diff, cmap='hot', origin='lower', norm=LogNorm(vmin=0.1, vmax=diff.max()))
axes[2].set_title('Injected Clusters Only')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## Run Detection and Retrieval Analysis

In [ ]:
# Run source detection
# For real analysis, use LSST's detection pipeline and pass the results here
detections = pipeline.run_detection(threshold=3.0)

print(f"Detected {len(detections)} sources")

In [ ]:
# Run retrieval analysis
retrieval = pipeline.run_retrieval(match_radius=5.0)

# Get summary statistics
stats = retrieval.get_summary_statistics()
print("\nRetrieval Summary:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

## Save Results

In [ ]:
# Save all results
pipeline.save_results()
print(f"\nResults saved to: {config.output_dir}")

In [ ]:
# Print 50% completeness limits
print("50% Completeness Limits:")
print(f"  Magnitude: {retrieval.get_50_percent_limit('magnitude'):.2f}")
print(f"  Half-light radius: {retrieval.get_50_percent_limit('r_half'):.2f} pixels")

In [ ]:
# Plot grid of all stamps
plot_stamp_grid(injection_info, n_cols=5, n_rows=4, show=True)

In [ ]:
# Completeness vs Magnitude
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Magnitude
ax = axes[0]
mag_bins = np.linspace(config.cluster_config.mag_min, config.cluster_config.mag_max, 12)
bin_centers, completeness, completeness_err = retrieval.compute_completeness('magnitude', bins=mag_bins)
ax.errorbar(bin_centers, completeness, yerr=completeness_err, fmt='o-', capsize=3)
ax.axhline(0.5, color='gray', linestyle='--', label='50%')
ax.axhline(0.9, color='gray', linestyle=':', label='90%')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Completeness')
ax.set_title('Completeness vs Magnitude')
ax.set_ylim(0, 1.05)
ax.legend()

# Half-light radius
ax = axes[1]
rh_bins = np.linspace(config.cluster_config.r_half_min, config.cluster_config.r_half_max, 10)
bin_centers, completeness, completeness_err = retrieval.compute_completeness('r_half', bins=rh_bins)
ax.errorbar(bin_centers, completeness, yerr=completeness_err, fmt='o-', capsize=3, color='orange')
ax.axhline(0.5, color='gray', linestyle='--')
ax.set_xlabel('Half-light Radius (pixels)')
ax.set_ylabel('Completeness')
ax.set_title('Completeness vs Size')
ax.set_ylim(0, 1.05)

# 2D completeness map
ax = axes[2]
result_2d = retrieval.compute_completeness_2d('magnitude', 'r_half', n_bins1=8, n_bins2=6)
im = ax.imshow(result_2d['completeness'].T, origin='lower', aspect='auto', cmap='RdYlGn',
               extent=[result_2d['bins1'][0], result_2d['bins1'][-1],
                       result_2d['bins2'][0], result_2d['bins2'][-1]],
               vmin=0, vmax=1)
ax.set_xlabel('Magnitude')
ax.set_ylabel('Half-light Radius (pixels)')
ax.set_title('2D Completeness Map')
plt.colorbar(im, ax=ax, label='Completeness')

plt.tight_layout()
plt.show()

### Option C: Load from file

In [ ]:
# Create one cluster with both methods
test_params = {
    'r_half': 15,
    'magnitude': 20,
    'profile_type': 'plummer',
}

# Smooth profile
smooth_cluster = create_cluster(
    method='smooth',
    r_half=test_params['r_half'],
    magnitude=test_params['magnitude'],
    profile_type=test_params['profile_type']
)

# Discrete stars
discrete_cluster = create_cluster(
    method='discrete',
    n_stars=200,
    r_half=test_params['r_half'],
    total_magnitude=test_params['magnitude'],
    profile_type=test_params['profile_type'],
    imf='kroupa',
    age_gyr=1.0,
    seed=42
)

print(f"Test cluster: r_half={test_params['r_half']}px, mag={test_params['magnitude']}")
print(f"\nSmooth: total_flux = {smooth_cluster.total_flux:.1f}")
print(f"Discrete: total_flux = {discrete_cluster.total_flux:.1f}, n_stars = {discrete_cluster.n_stars}")

In [ ]:
# Inject both and compare
test_position = (image.shape[0]//2, image.shape[1]//2)

# Inject smooth
_, _, smooth_stamps = inject_cluster(
    image, test_position, smooth_cluster,
    exposure=exposure, add_noise=False, return_stamps=True
)

# Inject discrete
_, _, discrete_stamps = inject_cluster(
    image, test_position, discrete_cluster,
    exposure=exposure, add_noise=False, return_stamps=True
)

# Compare
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Row 1: Smooth
for i, (key, title) in enumerate([('intrinsic', 'Intrinsic'), ('psf', 'PSF'), 
                                   ('convolved', 'Convolved'), ('final', 'Final')]):
    ax = axes[0, i]
    if smooth_stamps.get(key) is not None:
        img = smooth_stamps[key]
        if key == 'psf':
            im = ax.imshow(img, cmap='hot', origin='lower')
        else:
            im = ax.imshow(img, cmap='hot', origin='lower', 
                          norm=LogNorm(vmin=max(0.01, img.max()*1e-4), vmax=img.max()))
        ax.set_title(f'SMOOTH: {title}')
        plt.colorbar(im, ax=ax, shrink=0.7)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)

# Row 2: Discrete
for i, (key, title) in enumerate([('intrinsic', 'Intrinsic'), ('psf', 'PSF'), 
                                   ('convolved', 'Convolved'), ('final', 'Final')]):
    ax = axes[1, i]
    if discrete_stamps.get(key) is not None:
        img = discrete_stamps[key]
        if key == 'psf':
            im = ax.imshow(img, cmap='hot', origin='lower')
        else:
            im = ax.imshow(img, cmap='hot', origin='lower', 
                          norm=LogNorm(vmin=max(0.01, img.max()*1e-4), vmax=img.max()))
        ax.set_title(f'DISCRETE: {title}')
        plt.colorbar(im, ax=ax, shrink=0.7)
    else:
        ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)

plt.suptitle(f'Smooth vs Discrete Comparison (mag={test_params["magnitude"]}, r_half={test_params["r_half"]}px)', 
             fontsize=14)
plt.tight_layout()
plt.show()

---
## 8. Source Detection and Completeness Analysis

In [ ]:
# Run simple source detection
# NOTE: For real analysis, use LSST's detection pipeline!

DETECTION_THRESHOLD = 3.0  # sigma above background

print(f"Running source detection (threshold={DETECTION_THRESHOLD}σ)...")
detections = run_source_detection(injected_image, threshold=DETECTION_THRESHOLD)
print(f"✓ Found {len(detections)} sources")

In [ ]:
# Match injections with detections
MATCH_RADIUS = 5.0  # pixels

print(f"Matching with radius={MATCH_RADIUS} pixels...")
retrieval = ClusterRetrieval(injection_info, detections)
matches = retrieval.match_detections(match_radius=MATCH_RADIUS)

# Summary
stats = retrieval.get_summary_statistics()
print(f"\n" + "="*50)
print("RETRIEVAL SUMMARY")
print("="*50)
print(f"  Injected: {stats['n_injected']}")
print(f"  Detected: {stats['n_detected']}")
print(f"  Overall completeness: {stats['overall_completeness']:.1%}")
print(f"  50% mag limit: {stats['mag_50_limit']:.2f}")
print(f"  50% r_half limit: {stats['r_half_50_limit']:.2f} px")

In [ ]:
# Completeness curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Completeness vs Magnitude
ax = axes[0]
mag_bins = np.linspace(MAG_MIN, MAG_MAX, 12)
bin_centers, completeness, completeness_err = retrieval.compute_completeness('magnitude', bins=mag_bins)
ax.errorbar(bin_centers, completeness, yerr=completeness_err, fmt='o-', capsize=3, lw=2)
ax.axhline(0.5, color='red', linestyle='--', label='50%')
ax.axhline(0.9, color='green', linestyle=':', label='90%')
ax.set_xlabel('Magnitude')
ax.set_ylabel('Completeness')
ax.set_title('Completeness vs Magnitude')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(True, alpha=0.3)

# Completeness vs Size
ax = axes[1]
rh_bins = np.linspace(R_HALF_MIN, R_HALF_MAX, 10)
bin_centers, completeness, completeness_err = retrieval.compute_completeness('r_half', bins=rh_bins)
ax.errorbar(bin_centers, completeness, yerr=completeness_err, fmt='o-', capsize=3, lw=2, color='orange')
ax.axhline(0.5, color='red', linestyle='--')
ax.set_xlabel('Half-light Radius (pixels)')
ax.set_ylabel('Completeness')
ax.set_title('Completeness vs Size')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# 2D completeness map
ax = axes[2]
result_2d = retrieval.compute_completeness_2d('magnitude', 'r_half', n_bins1=8, n_bins2=6)
im = ax.imshow(result_2d['completeness'].T, origin='lower', aspect='auto', cmap='RdYlGn',
               extent=[result_2d['bins1'][0], result_2d['bins1'][-1],
                       result_2d['bins2'][0], result_2d['bins2'][-1]],
               vmin=0, vmax=1)
ax.set_xlabel('Magnitude')
ax.set_ylabel('Half-light Radius (pixels)')
ax.set_title('2D Completeness Map')
plt.colorbar(im, ax=ax, label='Completeness')

plt.suptitle(f'Completeness Analysis ({METHOD} method, N={N_CLUSTERS})', fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Save Results

In [ ]:
# Save results
import json
from datetime import datetime

OUTPUT_DIR = f'./injection_output_{METHOD}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save catalog
catalog_save = []
for info in injection_info:
    entry = {k: v for k, v in info.items() if k != 'stamps'}  # Don't save stamps
    entry = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in entry.items()}
    catalog_save.append(entry)

with open(os.path.join(OUTPUT_DIR, 'injection_catalog.json'), 'w') as f:
    json.dump(catalog_save, f, indent=2)

# Save retrieval results
retrieval.save_results(os.path.join(OUTPUT_DIR, 'retrieval_results.json'))

# Save configuration
config_save = {
    'method': METHOD,
    'n_clusters': N_CLUSTERS,
    'mag_range': [MAG_MIN, MAG_MAX],
    'r_half_range': [R_HALF_MIN, R_HALF_MAX],
    'profile_type': PROFILE_TYPE,
    'data_id': DATA_ID,
    'completeness_summary': stats
}
with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config_save, f, indent=2, default=float)

print(f"✓ Results saved to: {OUTPUT_DIR}")
print(f"  - injection_catalog.json")
print(f"  - retrieval_results.json")
print(f"  - config.json")

## Quick Reference: Configuration Options

### Cluster Methods
- `'smooth'`: Extended source with analytical profile (fast, good for distant clusters)
- `'discrete'`: Individual stars with IMF and spatial distribution (realistic, resolved clusters)

### Profile Types
- `'plummer'`: Simple analytical profile, good for many clusters
- `'king'`: Tidally truncated, good for globular clusters
- `'eff'`: Power-law profile, good for young clusters
- `'sersic'`: Generalized profile (n=1: exponential, n=4: de Vaucouleurs)

### IMF Options (for discrete stars)
- `'kroupa'`: Kroupa (2001) IMF
- `'chabrier'`: Chabrier (2003) IMF
- `'salpeter'`: Salpeter (1955) IMF